### Transformer

#### Imports

In [1]:
import torch
import torch.nn as nn
import math

#### Input embedding layer

<img src="diagrams/InputEmbedding.png" width="1000px">

In [2]:
class InputEmbeddings(nn.Module):
     def __init__(self, d_model: int, vocabulary_size: int) -> None:
          super().__init__()

          self.d_model = d_model
          self.vocabulary_size = vocabulary_size
          self.embedding = nn.Embedding(vocabulary_size, d_model)

     def forward(self, token_ids):
          return self.embedding(token_ids) * math.sqrt(self.d_model)

In [3]:
# Input embedding layer test
vocabulary_size = 100
d_model = 5
x = torch.tensor([
     [2, 43, 21],
])
e = InputEmbeddings(d_model, vocabulary_size)
e(x)

tensor([[[-2.0597,  1.7775, -1.8829,  3.3735, -6.1493],
         [ 1.9048,  2.2264, -3.2571, -3.1571,  2.6460],
         [ 1.5213, -0.6084,  0.0438, -0.4633, -2.7888]]],
       grad_fn=<MulBackward0>)

#### Positional Encoding

<img src="diagrams/PositionalEncoding.png" width="800px">

In [4]:
class PositionalEncoding(nn.Module):
     def __init__(self, sequenceLength: int, d_model: int, dropout: float):
          super().__init__()

          self.sequenceLength = sequenceLength
          self.d_model = d_model
          self.dropout = nn.Dropout(dropout)

          matrix = torch.zeros(sequenceLength, d_model)
          positions = torch.arange(0, sequenceLength, dtype = torch.float).unsqueeze(1)
          frequencyTerm = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0) / d_model))

          matrix[:, 0::2] = torch.sin(positions * frequencyTerm)
          matrix[:, 1::2] = torch.cos(positions * frequencyTerm)

          matrix = matrix.unsqueeze(0)
          self.register_buffer("matrix", matrix)

     def forward(self, inputEmbedding):
          inputEmbedding += self.matrix[:, :inputEmbedding.shape[1], :].requires_grad(False)
          return self.dropout(inputEmbedding)

#### Multi head attention

<img src="diagrams/MultiHeadAttention1.png" width="800px">

<img src="diagrams/MultiHeadAttention2.png" width="500px">

In [5]:
class MultiHeadAttention(nn.Module):
     def __init__(self, d_model: int, heads: int, dropout: float):
          super().__init__()

          self.d_model = d_model
          self.heads = heads
          assert d_model % heads == 0, "d_model is not divisible by heads"
          self.dimensionHead = d_model // heads

          self.wQ = nn.Linear(d_model, d_model, bias=False)
          self.wK = nn.Linear(d_model, d_model, bias=False)
          self.wV = nn.Linear(d_model, d_model, bias=False)
          self.wO = nn.Linear(d_model, d_model, bias=False)

          self.dropout = nn.Dropout(dropout)
          self.attentionMatrix = None
     
     @staticmethod
     def attention(query, key, value, mask=None, dropout=None):
          dimesnionKey = query.shape[-1]
          attentionMatrix = query @ key.transpose(-2, -1)
          attentionMatrix = attentionMatrix / math.sqrt(dimesnionKey)
          if mask is not None:
               attentionMatrix = attentionMatrix.masked_fill(mask == 0, -1e9)
          attentionWeights = attentionMatrix.softmax(dim=-1)
          if dropout is not None:
               attentionWeights = dropout(attentionWeights)
          output = attentionWeights @ value
          return output, attentionWeights
     
     def forward(self, q, k, v, mask=None):
          batchSize = q.shape[0]
          sequenceLength = q.shape[1]

          query = self.wQ(q)
          key = self.wK(k)
          value = self.wV(v)

          query = query.view(batchSize, sequenceLength, self.heads, self.dimensionHead)
          key = key.view(batchSize, sequenceLength, self.heads, self.dimensionHead)
          value = value.view(batchSize, sequenceLength, self.heads, self.dimensionHead)

          query = query.transpose(1,2)
          key = key.transpose(1,2)
          value = value.transpose(1,2)

          x, self.attentionMatrix = MultiHeadAttention.attention(
               query=query,
               key=key,
               value=value,
               mask=mask,
               dropout=self.dropout
          )

          x = x.transpose(1,2)
          x = x.contiguous()
          x = x.view(batchSize, sequenceLength, self.d_model)

          output = self.wO(x)
          return output

#### Layer Normalization

In [6]:
class LayerNormalization(nn.Module):
     def __init__(self, lastDimension: int, microValue: float=10**-8) -> None:
          super().__init__()

          self.lastDimension = lastDimension
          self.microValue = microValue
          self.gamma = nn.Parameter(torch.ones(lastDimension))
          self.beta = nn.Parameter(torch.zeros(lastDimension))

     def forward(self, input):
          mean = x.mean(dim=-1, keepdim=True)
          standardDeviation = x.std(dim=-1, keepdim=True)
          inputNormalised = (input - mean) / (standardDeviation + self.microValue)

          return self.gamma * inputNormalised + self.beta

#### Feed forward block

In [9]:
class FeedForwardBlock(nn.Module):
     def __init__(self, d_model: int, neurons: int, dropout: float) -> None:
          super().__init__()
          self.linear1 = nn.Linear(d_model, neurons)
          self.dropout = nn.Dropout(dropout)
          self.linear2 = nn.Linear(neurons, d_model)

     def forward(self, input):
          return self.linear2(self.dropout(torch.relu(self.linear1(input))))

#### Residual Connection

In [10]:
class ResidualConnection(nn.Module):
        def __init__(self, features: int, dropout: float) -> None:
            super().__init__()
            self.dropout = nn.Dropout(dropout)
            self.norm = LayerNormalization(features)
    
        def forward(self, x, sublayer):
            return x + self.dropout(sublayer(self.norm(x)))